# **Kaggle – DataTops®**
Tu TA ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Aspectos importantes
- Última submission:
    - Mañana: 17 de febrero a las 5pm
    - Tarde: 19 de febrero a las 5pm
- **Enlace de la competición**: https://www.kaggle.com/t/c5cc87b50c4b4770bdc8f5acbe15577d
- **Requisito**: Estar registrado en [Kaggle](https://www.kaggle.com/)

## Métrica:
El error cuadrático medio (RMSE, por sus siglas en inglés) es una medida de la desviación estándar de los residuos (errores de predicción). Los residuos representan la diferencia entre los valores observados y los valores predichos por el modelo. El RMSE indica qué tan dispersos están estos errores: cuanto menor es el RMSE, más cercanas están las predicciones a los valores reales. En otras palabras, el RMSE mide qué tan bien se ajusta la línea de regresión a los datos.


$$ RMSE = \sqrt{\frac{1}{n}\Sigma_{i=1}^{n}{\Big(\frac{d_i -f_i}{\sigma_i}\Big)^2}}$$


## 1. Librerías

In [95]:
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

import urllib.request
import matplotlib.pyplot as plt
import MyFunctions2 as mf    
import category_encoders as ce

In [96]:
# Semillas para reproducibilidad total
import random
import os
import numpy as np

random.seed(42)
os.environ["PYTHONHASHSEED"] = str(42)
np.random.seed(42)

# Para Optuna
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    # Cuando crees el estudio, añade seed=42:
    # study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
except ImportError:
    pass

# Para LightGBM y XGBoost, ya usas random_state=42 en los modelos, pero si usas funciones fuera de los modelos:
# import lightgbm as lgb
# lgb.train(..., seed=42)
# import xgboost as xgb
# xgb.train(..., seed=42)


## 2. Datos

In [97]:
# Para que funcione necesitas bajarte los archivos de datos de Kaggle
df = pd.read_csv("./data/train.csv")
X_pred = pd.read_csv("./data/test.csv")

### 2.1 Exploración de los datos

In [98]:
# Hacemos que nuestro índice sea el id
df.set_index("laptop_ID", inplace=True)
X_pred.set_index("laptop_ID", inplace=True)

y_test_pred = pd.read_csv("./data/test2.csv", sep=';')
y_test_pred.set_index("laptop_ID", inplace=True)    
y_test_pred["Price_in_euros"] = np.log1p(y_test_pred["Price_in_euros"])


In [99]:
df["Price_in_euros"] = np.log1p(df["Price_in_euros"])

### 2.3 Dividir X_train, X_test, y_train, y_test

In [100]:
train_set, test_set = train_test_split(df, test_size = 0.2, random_state = 42)

In [101]:
target_col = "Price_in_euros"
features = [c for c in train_set.columns if c != target_col]

y_train = train_set[target_col]
y_test = test_set[target_col]

X_train = train_set[features]
X_test = test_set[features]


In [102]:
def transformar_datos(df):

    mf.screen_resolution_split(df)
    mf.replace_value(df, "Ram", "GB", "", int)

    mf.replace_value(df, "Weight", "kg", "", float)
    mf.memory_split(df)

    mf.procesar_cpu_caotico(df, "Cpu")
    mf.gpu_split3(df)
    df.drop(["Product"], axis=1, inplace=True)

    return df


In [103]:
from sklearn.preprocessing import RobustScaler

X_train = transformar_datos(X_train)
X_test = transformar_datos(X_test)
X_pred = transformar_datos(X_pred)

# columnas numericas de X_train
#num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# columnas numéricas excluyendo binarias (solo continuas)
def get_numeric_continuous_cols(df):
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    bin_cols = [col for col in num_cols if df[col].nunique() == 2 and set(df[col].unique()) <= {0, 1}]
    cont_cols = [col for col in num_cols if col not in bin_cols]
    return cont_cols






In [104]:
# Crear nueva feature: píxeles por pulgada (ppi)
def add_ppi(df):
    # Asegúrate de tener las columnas correctas: 'screen_width', 'screen_height', 'Inches'
    if all(col in df.columns for col in ["screen_width", "screen_height", "Inches"]):
        df["ppi"] = ((df["screen_width"]**2 + df["screen_height"]**2)**0.5) / df["Inches"]
    return df

X_train = add_ppi(X_train)
X_test = add_ppi(X_test)
X_pred = add_ppi(X_pred)

# Puedes crear más features de interacción si lo deseas, por ejemplo:
# X_train["ram_cpu"] = X_train["Ram"] * X_train["cpu_marca"] (si cpu_marca es numérica)


In [105]:
num_cols = get_numeric_continuous_cols(X_train)

scaler = RobustScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])
X_pred[num_cols] = scaler.transform(X_pred[num_cols])

In [106]:
X_train

,Company,TypeName,OpSys,Full_HD,IPS_Panel,Touchscreen,4K_Ultra_HD,Retina_Display,Quad_HDplus,PPI,...,SSD,HDD,Flash_Storage,Hybrid,Memory_GB,cpu_marca,cpu_familia,cpu_ghz,brand_gpu,family_gpu
laptop_ID,,,,,,,,,,,,,,,,,,,,,
1118,HP,Workstation,Windows 7,1,1,0,0,0,0,-0.462315,...,0,1,0,0,0.682292,Intel,I7,0.111111,AMD,FirePro
153,Dell,Gaming,Windows 10,1,0,0,0,0,0,0.000000,...,1,0,0,0,0.015625,Intel,I7,0.333333,Nvidia,GeForce
275,Apple,Ultrabook,macOS,0,1,0,0,1,0,2.857620,...,1,0,0,0,0.015625,Intel,I5,0.444444,Intel,Iris
1100,HP,Notebook,Windows 7,1,0,0,0,0,0,0.537685,...,0,1,0,0,0.000000,Intel,I5,-0.222222,Intel,HD Graphics
131,Dell,Notebook,Windows 10,1,0,0,0,0,0,-0.462315,...,1,1,0,0,2.348958,Intel,I7,-0.777778,AMD,Radeon
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
578,HP,Notebook,Windows 10,0,0,0,0,0,0,-0.975411,...,0,1,0,0,2.015625,Intel,PENTIUM,-1.000000,Intel,HD Graphics
996,Lenovo,Notebook,Windows 10,1,0,0,0,0,0,0.000000,...,1,0,0,0,-0.317708,AMD,Other,1.222222,AMD,Radeon
770,Dell,Ultrabook,Windows 10,1,0,0,0,0,0,1.166775,...,1,0,0,0,-0.317708,Intel,I7,0.333333,Intel,HD Graphics


In [107]:
for col in X_train.select_dtypes(include='object').columns:
    print(f"{col}: {X_train[col].nunique()} categorías")

Company: 17 categorías
TypeName: 6 categorías
OpSys: 9 categorías
cpu_marca: 2 categorías
cpu_familia: 9 categorías
brand_gpu: 3 categorías
family_gpu: 7 categorías


In [108]:
#--- Codificación manual sin ColumnTransformer y sin copias ---
from sklearn.preprocessing import OneHotEncoder
import category_encoders as ce
import pandas as pd

onehot_cols = ["TypeName", "cpu_marca", "brand_gpu"]
targetencoder_cols = ["Company", "OpSys", "cpu_familia", "family_gpu"]

# Asegurar consistencia de categorías en OneHotEncoder
for col in onehot_cols:
    top_cats = X_train[col].value_counts().nlargest(10).index
    X_train[col] = X_train[col].where(X_train[col].isin(top_cats), 'Other')
    X_test[col] = X_test[col].where(X_test[col].isin(top_cats), 'Other')
    X_pred[col] = X_pred[col].where(X_pred[col].isin(top_cats), 'Other')

# Target Encoding (modifica X_train, X_test, X_pred directamente)
target_encoder = ce.TargetEncoder(cols=targetencoder_cols)
X_train[targetencoder_cols] = target_encoder.fit_transform(X_train[targetencoder_cols], y_train)
X_test[targetencoder_cols] = target_encoder.transform(X_test[targetencoder_cols])
X_pred[targetencoder_cols] = target_encoder.transform(X_pred[targetencoder_cols])

# OneHot Encoding (solo para columnas onehot_cols)
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
X_train_ohe = pd.DataFrame(
    ohe.fit_transform(X_train[onehot_cols]),
    columns=ohe.get_feature_names_out(onehot_cols),
    index=X_train.index
    )
X_test_ohe = pd.DataFrame(
    ohe.transform(X_test[onehot_cols]),
    columns=ohe.get_feature_names_out(onehot_cols),
    index=X_test.index
    )
X_pred_ohe = pd.DataFrame(
    ohe.transform(X_pred[onehot_cols]),
    columns=ohe.get_feature_names_out(onehot_cols),
    index=X_pred.index
    )

# Eliminar columnas originales onehot_cols (ya codificadas)
X_train.drop(columns=onehot_cols, inplace=True)
X_test.drop(columns=onehot_cols, inplace=True)
X_pred.drop(columns=onehot_cols, inplace=True)

# Concatenar los resultados
X_train = pd.concat([X_train, X_train_ohe], axis=1)
X_test = pd.concat([X_test, X_test_ohe], axis=1)
X_pred = pd.concat([X_pred, X_pred_ohe], axis=1)

c:\Users\jvend\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [109]:
# Eliminar variables con baja correlación respecto al target
# Solo considerar columnas numéricas para la correlación
def drop_low_correlation_features(X, y, threshold=0.05):
    X_numeric = X.select_dtypes(include=[np.number])
    corrs = X_numeric.corrwith(y).abs()
    low_corr_cols = corrs[corrs < threshold].index.tolist()
    print(f"Eliminando {len(low_corr_cols)} columnas con correlación < {threshold}: {low_corr_cols}")
    return X.drop(columns=low_corr_cols), low_corr_cols

X_train, dropped_low_corr = drop_low_correlation_features(X_train, y_train, threshold=0.05)
X_test = X_test.drop(columns=dropped_low_corr)
X_pred = X_pred.drop(columns=dropped_low_corr)


Eliminando 1 columnas con correlación < 0.05: ['Hybrid']


In [110]:
# Calcular la correlación de cada feature con el target
correlation = X_train.corrwith(y_train)
correlation.sort_values(ascending=False, inplace=True)
correlation

cpu_familia             0.770458
Ram_GB                  0.677051
SSD                     0.592999
cpu_ghz                 0.497208
PPI                     0.490018
TypeName_Gaming         0.375279
family_gpu              0.373480
Full_HD                 0.372853
OpSys                   0.349454
Company                 0.346207
brand_gpu_Nvidia        0.341210
TypeName_Ultrabook      0.314904
IPS_Panel               0.292776
Touchscreen             0.234763
4K_Ultra_HD             0.228327
cpu_marca_Intel         0.223724
TypeName_Workstation    0.177832
Memory_GB               0.174840
Quad_HDplus             0.145903
Weight_kg               0.140551
Retina_Display          0.101851
HDD                    -0.160063
brand_gpu_Intel        -0.195111
TypeName_Netbook       -0.216911
Flash_Storage          -0.327330
TypeName_Notebook      -0.569807
dtype: float64

In [111]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Entrenar el modelo Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

def predecir(model, X_test, y_test):
    # Predecir sobre el test
    y_pred = model.predict(X_test)

    # Calcular RMSE en escala logarítmica
    rmse_log = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"RMSE (log1p): {rmse_log:.4f}")

    # Calcular RMSE en escala original (deshacer log1p)
    y_test_exp = np.expm1(y_test)
    y_pred_exp = np.expm1(y_pred)
    rmse_original = np.sqrt(mean_squared_error(y_test_exp, y_pred_exp))
    print(f"RMSE (escala original): {rmse_original:.2f}")

In [112]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error
import numpy as np

# Búsqueda de hiperparámetros ampliada para RandomForest
param_grid = {
    'n_estimators': [100, 200, 400],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    #'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

rf = RandomForestRegressor(random_state=42)
grid_search_rf = GridSearchCV(
    rf, param_grid, cv=3, scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=2
)
grid_search_rf.fit(X_train, y_train)


Fitting 3 folds for each of 384 candidates, totalling 1152 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'bootstrap': [True, False], 'max_depth': [None, 10, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation 

In [113]:
import optuna
import xgboost as xgb
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error

# Optuna para XGBoost

def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'random_state': 42
    }
    model = xgb.XGBRegressor(**params)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error').mean()
    return score

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=75)

print("Mejores hiperparámetros XGBoost:", study.best_params)

# Entrenar modelo final con los mejores hiperparámetros
best_params = study.best_params
model_xgb_optuna = xgb.XGBRegressor(**best_params)
model_xgb_optuna.fit(X_train, y_train)




Mejores hiperparámetros XGBoost: {'max_depth': 8, 'learning_rate': 0.05079385755978383, 'n_estimators': 194, 'subsample': 0.7626333554435598, 'colsample_bytree': 0.6272353029327429, 'gamma': 0.008166960616973029, 'reg_alpha': 0.07754407125481011, 'reg_lambda': 1.723434264771729, 'min_child_weight': 1}


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.6272353029327429
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import

In [116]:
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

def objective_lgb(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 100, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 30),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
        'random_state': 42
    }
    model = lgb.LGBMRegressor(**params)
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='neg_root_mean_squared_error').mean()
    return score

study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(objective_lgb, n_trials=50)

print("Mejores hiperparámetros LightGBM:", study_lgb.best_params)

best_params_lgb = study_lgb.best_params
model_lgb_optuna = lgb.LGBMRegressor(**best_params_lgb)
model_lgb_optuna.fit(X_train, y_train)


C:\Users\jvend\AppData\Local\Temp\ipykernel_35232\988084864.py:7: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'num_leaves': trial.suggest_int('num_leaves', 20, 100, 150),
c:\Users\jvend\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\distributions.py:684: UserWarning: The distribution is specified by [20, 100] and step=150, but the range is not divisible by `step`. It will be replaced with [20, 20].
  optuna_warn(
C:\Users\jvend\AppData\Local\Temp\ipykernel_35232\988084864.py:7: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as k

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0,000077 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 187
[LightGBM] [Info] Number of data points in the train set: 486, number of used features: 20
[LightGBM] [Info] Start training from score 6,826433
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

C:\Users\jvend\AppData\Local\Temp\ipykernel_35232\988084864.py:7: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'num_leaves': trial.suggest_int('num_leaves', 20, 100, 150),
c:\Users\jvend\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\distributions.py:684: UserWarning: The distribution is specified by [20, 100] and step=150, but the range is not divisible by `step`. It will be replaced with [20, 20].
  optuna_warn(
C:\Users\jvend\AppData\Local\Temp\ipykernel_35232\988084864.py:7: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as k

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0,000147 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 189
[LightGBM] [Info] Number of data points in the train set: 486, number of used features: 20
[LightGBM] [Info] Start training from score 6,819085
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

C:\Users\jvend\AppData\Local\Temp\ipykernel_35232\988084864.py:7: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'num_leaves': trial.suggest_int('num_leaves', 20, 100, 150),
c:\Users\jvend\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\distributions.py:684: UserWarning: The distribution is specified by [20, 100] and step=150, but the range is not divisible by `step`. It will be replaced with [20, 20].
  optuna_warn(
C:\Users\jvend\AppData\Local\Temp\ipykernel_35232\988084864.py:7: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as k

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0,000134 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 197
[LightGBM] [Info] Number of data points in the train set: 486, number of used features: 25
[LightGBM] [Info] Start training from score 6,826433
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

C:\Users\jvend\AppData\Local\Temp\ipykernel_35232\988084864.py:7: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'num_leaves': trial.suggest_int('num_leaves', 20, 100, 150),
c:\Users\jvend\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\distributions.py:684: UserWarning: The distribution is specified by [20, 100] and step=150, but the range is not divisible by `step`. It will be replaced with [20, 20].
  optuna_warn(
C:\Users\jvend\AppData\Local\Temp\ipykernel_35232\988084864.py:7: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as k

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

C:\Users\jvend\AppData\Local\Temp\ipykernel_35232\988084864.py:7: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'num_leaves': trial.suggest_int('num_leaves', 20, 100, 150),
c:\Users\jvend\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\distributions.py:684: UserWarning: The distribution is specified by [20, 100] and step=150, but the range is not divisible by `step`. It will be replaced with [20, 20].
  optuna_warn(


,boosting_type,'gbdt'
,num_leaves,20
,max_depth,9
,learning_rate,0.04591040374887076
,n_estimators,214
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,7


In [117]:
predecir(grid_search_rf, X_pred, y_test_pred)
# Evaluación en test
predecir(model_xgb_optuna, X_pred, y_test_pred)
predecir(model_lgb_optuna, X_pred, y_test_pred)

RMSE (log1p): 0.2160
RMSE (escala original): 348.19
RMSE (log1p): 0.1954
RMSE (escala original): 310.89
RMSE (log1p): 0.1987
RMSE (escala original): 307.03


In [ ]:
# CatBoostRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error

# Entrenamiento básico (puedes ajustar hiperparámetros)
catboost_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    loss_function='RMSE',
    random_state=42,
    verbose=100
)
catboost_model.fit(X_train, y_train)

# Predicción y evaluación
y_pred_cat = catboost_model.predict(X_test)
rmse_log_cat = np.sqrt(mean_squared_error(y_test, y_pred_cat))
print(f"RMSE CatBoost (log1p): {rmse_log_cat:.4f}")
y_pred_cat_exp = np.expm1(y_pred_cat)
y_test_exp = np.expm1(y_test)
rmse_original_cat = np.sqrt(mean_squared_error(y_test_exp, y_pred_cat_exp))
print(f"RMSE CatBoost (escala original): {rmse_original_cat:.2f}")

In [120]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error

def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 40),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),
        'random_state': 42
    }
    model = RandomForestRegressor(**params)
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='neg_root_mean_squared_error').mean()
    return score

study_rf = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(objective_rf, n_trials=50)

print("Mejores hiperparámetros RandomForest (Optuna):", study_rf.best_params)

best_params_rf = study_rf.best_params
model_rf_optuna = RandomForestRegressor(**best_params_rf)
model_rf_optuna.fit(X_train, y_train)


Mejores hiperparámetros RandomForest (Optuna): {'n_estimators': 225, 'max_depth': 34, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'bootstrap': False}


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",225
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",34
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsampl

In [121]:
predecir(model_rf_optuna, X_pred, y_test_pred)

RMSE (log1p): 0.2031
RMSE (escala original): 329.74


In [118]:
# Ensemble: promedio de XGBoost, RandomForest y LightGBM (Optuna)
# Asegúrate de tener entrenados: model_xgb_optuna, grid_search_rf, model_lgb_optuna

pred_xgb = model_xgb_optuna.predict(X_pred)
pred_rf = grid_search_rf.predict(X_pred)
pred_lgb = model_lgb_optuna.predict(X_pred)

ensemble_pred_3 = (pred_xgb + pred_lgb) / 2

y_test_exp = np.expm1(y_test_pred)
y_pred_exp = np.expm1(ensemble_pred_3)
rmse_original = np.sqrt(mean_squared_error(y_test_exp, y_pred_exp))
print(f"RMSE Ensemble (2 modelos): {rmse_original:.2f}")

RMSE Ensemble (2 modelos): 307.06
